# Malware Embedding & Clustering

## What This Is
Malware families share behavioral patterns. Dynamic analysis — running malware in a sandbox and recording API call sequences — produces temporal traces that reveal behavioral families. Embedding these traces enables family clustering and variant detection.

**API call sequences**: Malware executes characteristic sequences: CreateFile → WriteFile → SetFileAttributes (file dropper), or CreateRemoteThread → VirtualAllocEx → WriteProcessMemory (process injection).

**Embedding approaches**:
- Word2Vec on API tokens: treats API calls as words in a sentence
- LSTM encoder: captures temporal ordering and dependencies
- Transformer encoder: parallel, attention-based sequence modeling

**Clustering**: After embedding, UMAP + HDBSCAN identifies behavioral families without requiring labels.

In [ ]:
import numpy as np
from typing import List, Dict, Tuple

np.random.seed(42)

# API call vocabulary (common Windows API calls in malware)
API_VOCAB = [
    # File operations
    'CreateFile', 'ReadFile', 'WriteFile', 'DeleteFile', 'CopyFile',
    # Process operations
    'CreateProcess', 'OpenProcess', 'TerminateProcess', 'CreateRemoteThread',
    'VirtualAllocEx', 'WriteProcessMemory',
    # Registry
    'RegOpenKey', 'RegSetValue', 'RegCreateKey', 'RegDeleteKey',
    # Network
    'socket', 'connect', 'send', 'recv', 'InternetOpen', 'URLDownload',
    # Crypto
    'CryptEncrypt', 'CryptGenKey', 'CryptAcquireContext',
    # System
    'GetSystemInfo', 'SetFileTime', 'GetTempPath', 'CreateMutex',
    'LoadLibrary', 'GetProcAddress',
]
API2IDX = {api: i for i, api in enumerate(API_VOCAB)}

# Malware families with characteristic API sequences
FAMILY_TEMPLATES = {
    'ransomware': [
        ['CreateFile', 'CryptAcquireContext', 'CryptGenKey', 'CryptEncrypt',
         'WriteFile', 'DeleteFile', 'RegSetValue'],
    ],
    'trojan_injector': [
        ['OpenProcess', 'VirtualAllocEx', 'WriteProcessMemory', 'CreateRemoteThread'],
    ],
    'downloader': [
        ['InternetOpen', 'URLDownload', 'CreateFile', 'WriteFile', 'CreateProcess'],
    ],
    'keylogger': [
        ['SetWindowsHook', 'GetSystemInfo', 'WriteFile', 'send'],
    ],
}

def generate_api_trace(family: str, trace_len: int = 30, noise: float = 0.2) -> List[str]:
    """Generate a synthetic API call trace for a malware family."""
    template = FAMILY_TEMPLATES[family][0]
    trace = []
    template_idx = 0
    
    for i in range(trace_len):
        if np.random.random() < noise:
            # Noise: random API call
            trace.append(np.random.choice(API_VOCAB))
        else:
            # Template API call (with cycling)
            trace.append(template[template_idx % len(template)])
            template_idx += 1
    return trace

# Generate traces for 4 families
traces = []
families = []
for family in FAMILY_TEMPLATES:
    for _ in range(100):
        traces.append(generate_api_trace(family, trace_len=30, noise=0.25))
        families.append(family)

print(f'Generated {len(traces)} malware traces across {len(FAMILY_TEMPLATES)} families')
print(f'Sample ransomware trace: {traces[0][:8]}...')


In [ ]:
# Embed API traces using TF-IDF style bag-of-APIs
# Then visualize separation between families

def trace_to_bow(trace: List[str], vocab: List[str]) -> np.ndarray:
    """Convert API trace to bag-of-words vector."""
    vec = np.zeros(len(vocab))
    for api in trace:
        if api in API2IDX:
            vec[API2IDX[api]] += 1
    return vec / (len(trace) + 1)

def trace_to_ngram(trace: List[str], n: int = 2) -> Dict[Tuple, int]:
    """Extract n-gram API sequences."""
    ngrams = {}
    for i in range(len(trace) - n + 1):
        gram = tuple(trace[i:i+n])
        ngrams[gram] = ngrams.get(gram, 0) + 1
    return ngrams

# Embed traces
X_bow = np.array([trace_to_bow(t, API_VOCAB) for t in traces])

# Simple dimensionality reduction via PCA
X_centered = X_bow - X_bow.mean(0)
cov = X_centered.T @ X_centered / len(X_centered)
vals, vecs = np.linalg.eigh(cov)
top_vecs = vecs[:, np.argsort(vals)[-2:]]  # top 2 PCs
X_2d = X_centered @ top_vecs

# Report cluster separation
family_list = list(FAMILY_TEMPLATES.keys())
colors = {f: i for i, f in enumerate(family_list)}

print('Malware Family Separation (2D PCA)')
print('=' * 50)
for family in family_list:
    mask = [f == family for f in families]
    pts = X_2d[mask]
    print(f'  {family:20s}: centroid = ({pts[:,0].mean():+.3f}, {pts[:,1].mean():+.3f})')

# Inter-cluster distance
centroids = {f: X_2d[[i for i,fam in enumerate(families) if fam==f]].mean(0) for f in family_list}
print('\nPairwise centroid distances:')
for i, f1 in enumerate(family_list):
    for f2 in family_list[i+1:]:
        d = np.linalg.norm(centroids[f1] - centroids[f2])
        print(f'  {f1} vs {f2}: {d:.4f}')


In [ ]:
# LSTM-style sequential embedding (simplified)
# Key insight: n-gram overlap between families reveals behavioral similarity

def compute_bigram_similarity(traces1: List[List[str]], traces2: List[List[str]]) -> float:
    """Average bigram Jaccard similarity between two trace sets."""
    sim_scores = []
    for t1 in traces1[:20]:
        for t2 in traces2[:20]:
            bg1 = set(tuple(t1[i:i+2]) for i in range(len(t1)-1))
            bg2 = set(tuple(t2[i:i+2]) for i in range(len(t2)-1))
            if bg1 | bg2:
                sim_scores.append(len(bg1 & bg2) / len(bg1 | bg2))
    return np.mean(sim_scores) if sim_scores else 0.0

family_traces = {
    f: [t for t, fam in zip(traces, families) if fam == f]
    for f in family_list
}

print('Behavioral Similarity Matrix (Bigram Jaccard):')
print(f'{'':20s}', end='')
for f in family_list:
    print(f'{f[:10]:>12s}', end='')
print()

for f1 in family_list:
    print(f'{f1:20s}', end='')
    for f2 in family_list:
        sim = compute_bigram_similarity(family_traces[f1], family_traces[f2])
        print(f'{sim:12.3f}', end='')
    print()

print('\nHigh diagonal, low off-diagonal = good family separation')
print('In production: replace with LSTM/Transformer encoder + UMAP + HDBSCAN clustering')
